In [ ]:
!pip install transformers datasets evaluate scikit-learn accelerate

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from datasets import load_dataset
ds = load_dataset("hajili/azerbaijani_review_sentiment_classification")

ds = ds.filter(lambda x: x["content"] is not None and len(x["content"].strip()) > 0)

def to_label(x):
    s = x["score"]
    return {"label": 0 if s <= 2 else (1 if s == 3 else 2)}
ds = ds.map(to_label)
ds = ds.rename_column("content", "text")

from collections import Counter
print(Counter(ds["train"]["label"]))

from datasets import concatenate_datasets

def balanced(split, n_per_class):
    parts = []
    for c in [0, 1, 2]:
        d = split.filter(lambda x: x["label"] == c).shuffle(seed=42)
        parts.append(d.select(range(min(n_per_class, len(d)))))
    return concatenate_datasets(parts).shuffle(seed=42)

ds["train"] = balanced(ds["train"], 20000)
ds["test"]  = balanced(ds["test"], 600)
print(Counter(ds["train"]["label"]), Counter(ds["test"]["label"]))
print(ds)

README.md:   0%|          | 0.00/222 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/127537 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/31885 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/127534 [00:00<?, ? examples/s]

Filter:   0%|          | 0/31885 [00:00<?, ? examples/s]

Filter:   0%|          | 0/31885 [00:00<?, ? examples/s]

Filter:   0%|          | 0/31885 [00:00<?, ? examples/s]

Counter({2: 20000, 0: 16395, 1: 2763}) Counter({0: 600, 1: 600, 2: 600})
DatasetDict({
    train: Dataset({
        features: ['text', 'score', 'upvotes', 'label'],
        num_rows: 39158
    })
    test: Dataset({
        features: ['text', 'score', 'upvotes', 'label'],
        num_rows: 1800
    })
})


In [ ]:
split = ds["train"].train_test_split(test_size=0.1, seed=42)
ds["train"], ds["validation"] = split["train"], split["test"]

from transformers import AutoTokenizer
name = "xlm-roberta-base"
tok = AutoTokenizer.from_pretrained(name)
def prep(batch):
    return tok(batch["text"], truncation=True, max_length=128)
ds = ds.map(prep, batched=True)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/35242 [00:00<?, ? examples/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/3916 [00:00<?, ? examples/s]

In [ ]:
import numpy as np, evaluate
from transformers import (AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding)

id2label = {0: "negative", 1: "neutral", 2: "positive"}
model = AutoModelForSequenceClassification.from_pretrained(
    name, num_labels=3, id2label=id2label,
    label2id={v: k for k, v in id2label.items()})

f1 = evaluate.load("f1")
def metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return f1.compute(predictions=preds, references=p.label_ids, average="macro")

args = TrainingArguments(
    output_dir="out", num_train_epochs=3,
    per_device_train_batch_size=16, learning_rate=2e-5,
    eval_strategy="epoch", fp16=True, report_to="none")

trainer = Trainer(model=model, args=args,
    train_dataset=ds["train"], eval_dataset=ds["test"],
    data_collator=DataCollatorWithPadding(tok),
    compute_metrics=metrics)
trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

pred = trainer.predict(ds["test"])
y_pred = np.argmax(pred.predictions, axis=1)
y_true = pred.label_ids
names = ["negative", "neutral", "positive"]

print(classification_report(y_true, y_pred, target_names=names, digits=3))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=names).plot(cmap="Blues")
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
df = pd.DataFrame({
    "text": ds["test"]["text"],
    "gerçek": [names[i] for i in y_true],
    "tahmin": [names[i] for i in y_pred],
})
yanlis = df[df["gerçek"] != df["tahmin"]]
print(len(yanlis), "yanlış /", len(df))
yanlis.sample(15, random_state=1)

In [ ]:
model.push_to_hub("azerbaijani_sentiment_analysis")
tok.push_to_hub("azerbaijani_sentiment_analysis")

In [ ]:
!pip install gradio
import gradio as gr
from transformers import pipeline

clf = pipeline("text-classification", model="twnar/azerbaijani_sentiment_analysis", top_k=None)

def predict(text):
    if not text.strip():
        return {}
    return {r["label"]: r["score"] for r in clf(text)[0]}
gr.Interface(
    fn=predict,
    inputs=gr.Textbox(label="Azərbaycanca cümlə yaz", lines=3),
    outputs=gr.Label(num_top_classes=3),
    title="Azerbaijani Sentiment Analysis",
    examples=["Bu tətbiq çox yaxşıdır!", "Proqrama girmək olmur.", "Xidmət çox pis idi."],
).launch(share=True)